# 02 — Single-Indicator IC Analysis (v2: 3yr window + intraday-only)

**Phase 2 deliverable** from `docs/research_plan.md`.

v1 ran on 2yr (Jan-2023 → Dec-2024), a strong bull period. All survivors came out trend-positive at 1-day horizon — but we couldn't distinguish *real momentum signal* from *bull-market regime bias*. This rerun fixes both:

1. **Extend window to 3 years** (Jan-2022 → Dec-2024) to include the 2022 bear and 2023-24 rally — IC averaged across regimes is more honest.
2. **Add intraday-only IC** — forward returns where bar `t+n` is on the same calendar date as `t`. Strips out overnight gap noise; lets us see whether mean-reversion lives at short horizons even when daily returns trend.

Indicators unchanged from v1 (`indicators.py`).

In [ ]:
# region imports
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from indicators import compute_all, ALL as INDICATOR_FNS
from ic_calculator import (
    compute_ic, ic_verdict, quantile_test, rolling_ic,
    intraday_forwards, compute_ic_from_forwards,
)
# endregion

qb = QuantBook()
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 140)

TICKERS = ['SPY', 'QQQ', 'IWM']
STANDARD_PERIODS = (1, 4, 24, 96)        # 15m, 1h, 1d, 4d
INTRADAY_PERIODS = (1, 2, 4, 8)          # 15m, 30m, 1h, 2h — all within a 26-bar session

END   = datetime(2024, 12, 31)
START = END - timedelta(days=365 * 3)    # 3 years

## 1. Fetch and resample to 15-min (3 years)

In [ ]:
symbols = {t: qb.add_equity(t, Resolution.MINUTE).symbol for t in TICKERS}
history = qb.history(list(symbols.values()), START, END, Resolution.MINUTE)
print(f'minute rows: {len(history):,}')

def resample_15min(df: pd.DataFrame) -> pd.DataFrame:
    return df.resample('15min').agg({
        'open':   'first',
        'high':   'max',
        'low':    'min',
        'close':  'last',
        'volume': 'sum',
    }).dropna(subset=['open'])

bars_15m = {t: resample_15min(history.loc[sym]) for t, sym in symbols.items()}
for t, df in bars_15m.items():
    print(f'  {t}: {len(df):,} 15-min bars  ({df.index.min().date()} → {df.index.max().date()})')

## 2. Compute indicators on every symbol

In [ ]:
indicators_by_sym = {t: compute_all(df) for t, df in bars_15m.items()}
prices_by_sym     = {t: df['close'] for t, df in bars_15m.items()}
returns_by_sym    = {t: df['close'].pct_change() for t, df in bars_15m.items()}

# Precompute intraday forwards once per symbol — heavy step at 3yr × 3 symbols
intraday_fwd_by_sym = {t: intraday_forwards(prices_by_sym[t], INTRADAY_PERIODS)
                       for t in TICKERS}
for t in TICKERS:
    n = len(prices_by_sym[t])
    for p, s in intraday_fwd_by_sym[t].items():
        print(f'  {t} intraday fwd p={p:>2}: {s.notna().sum():>6,}/{n:,} '
              f'({s.notna().mean()*100:.1f}% non-null)')

## 3. Standard IC — cross-day forward returns (v1 measure)

Same as v1 for direct comparison. Forward windows can cross day boundaries → includes overnight gaps. Periods: 1, 4, 24, 96 bars.

In [ ]:
records = []
for sym, ind_df in indicators_by_sym.items():
    rets = returns_by_sym[sym]
    for name in ind_df.columns:
        for period, r in compute_ic(ind_df[name], rets, periods=STANDARD_PERIODS).items():
            records.append({'symbol': sym, 'indicator': name, 'period': period,
                            'ic': r.ic, 'pvalue': r.pvalue, 'n': r.n})
ic_std_long = pd.DataFrame(records)

avg_ic_std = (ic_std_long.groupby(['indicator', 'period'])['ic']
              .mean().unstack('period').reindex(INDICATOR_FNS.keys()))
print('Standard IC — mean across SPY/QQQ/IWM:')
print(avg_ic_std.to_string())

best_p_std  = avg_ic_std.abs().idxmax(axis=1)
best_ic_std = pd.Series({n: avg_ic_std.loc[n, best_p_std[n]] for n in avg_ic_std.index})
best_std    = pd.DataFrame({
    'best_period': best_p_std,
    'best_ic':     best_ic_std,
    'verdict':     best_ic_std.apply(ic_verdict),
})
print('\nStandard best-horizon summary:')
print(best_std.to_string())

## 4. Intraday-only IC — forward returns masked at day boundaries

For each bar `t`, the forward return at period `n` is `price[t+n]/price[t] - 1` only if `t+n` falls in the same trading day; otherwise NaN. This strips overnight gaps. Periods: 1, 2, 4, 8 bars (all within a 26-bar session).

In [ ]:
records = []
for sym, ind_df in indicators_by_sym.items():
    fwds = intraday_fwd_by_sym[sym]
    for name in ind_df.columns:
        for period, r in compute_ic_from_forwards(ind_df[name], fwds).items():
            records.append({'symbol': sym, 'indicator': name, 'period': period,
                            'ic': r.ic, 'pvalue': r.pvalue, 'n': r.n})
ic_id_long = pd.DataFrame(records)

avg_ic_id = (ic_id_long.groupby(['indicator', 'period'])['ic']
             .mean().unstack('period').reindex(INDICATOR_FNS.keys()))
print('Intraday-only IC — mean across SPY/QQQ/IWM:')
print(avg_ic_id.to_string())

best_p_id  = avg_ic_id.abs().idxmax(axis=1)
best_ic_id = pd.Series({n: avg_ic_id.loc[n, best_p_id[n]] for n in avg_ic_id.index})
best_id    = pd.DataFrame({
    'best_period': best_p_id,
    'best_ic':     best_ic_id,
    'verdict':     best_ic_id.apply(ic_verdict),
})
print('\nIntraday best-horizon summary:')
print(best_id.to_string())

## 5. Side-by-side: standard vs intraday

If an indicator has **opposite signs** between the two measures, it has different intraday vs cross-day behavior — usually means the daily-horizon signal is dominated by overnight drift, while intraday is closer to noise (or a real reversal).

If both are same-sign with similar magnitude, the signal is real and survives both lenses.

In [ ]:
compare = pd.DataFrame({
    'std_ic':       best_std['best_ic'],
    'std_period':   best_std['best_period'],
    'std_verdict':  best_std['verdict'],
    'id_ic':        best_id['best_ic'],
    'id_period':    best_id['best_period'],
    'id_verdict':   best_id['verdict'],
})
compare['sign_flip'] = np.sign(compare['std_ic']) != np.sign(compare['id_ic'])
print('Side-by-side (rows=indicator):')
print(compare.to_string())

# Union of survivors — non-reject in either measure
survivors = sorted(set(best_std[best_std.verdict != 'reject'].index)
                   | set(best_id[best_id.verdict != 'reject'].index))
print(f'\nsurvivors (either measure): {survivors}')
print(f'sign-flipped indicators:    {compare[compare.sign_flip].index.tolist()}')

## 6. Quantile tests for survivors

Standard (period=24, ≈1 day) bucket means. A real factor produces a monotonic ramp; flat or U-shape means the IC is a fluke or non-monotone.

In [ ]:
for name in survivors:
    print(f'\n── {name} — quantile test (period=24, ≈ 1 day) ──')
    for sym in TICKERS:
        q = quantile_test(indicators_by_sym[sym][name], returns_by_sym[sym],
                          n_quantiles=5, period=24)
        print(f'  {sym}:  ' + '  '.join(
            f'Q{i}={q.loc[i,"mean"]:+.5f}' for i in q.index
        ))

## 7. Rolling IC stability (standard, SPY, 1-month window)

Frequent zero-crossings = the indicator works in some periods and reverses in others = not actionable.

In [ ]:
if survivors:
    fig, axes = plt.subplots(len(survivors), 1, figsize=(11, 2.4 * len(survivors)), squeeze=False)
    rets_spy = returns_by_sym['SPY']
    for ax, name in zip(axes[:, 0], survivors):
        ric = rolling_ic(indicators_by_sym['SPY'][name], rets_spy, window=2016, period=24)
        ax.plot(ric.index, ric.values, linewidth=0.9)
        ax.axhline(0, color='red', alpha=0.5, linewidth=0.8)
        ax.set_title(f'SPY rolling 1-month IC — {name} (period=24)', fontsize=10)
        ax.set_ylim(-0.3, 0.3)
    plt.tight_layout()
else:
    print('No survivors — skipping rolling IC plot.')

## 8. Final verdict

Reading the tables in order:

- **Section 3**: does adding the 2022 bear change the headline signs? If v1's all-positive IC table now shows some negatives, regime was the explanation.
- **Section 4**: are intraday IC signs different from cross-day signs? If yes → the 1-day signal was overnight drift, not in-session pattern.
- **Section 5**: which indicators flipped signs? They're regime/horizon-sensitive — handle with care in Phase 3.
- **Section 6**: which survivors have monotonic buckets? Those are the cleanest signals to build a strategy around.
- **Section 7**: which survivors have stable rolling-IC signs? Those work across market periods.

An indicator that passes all four (non-reject in section 4 OR 5, monotonic in 6, stable in 7) earns a Phase 3 strategy slot. The original spec's combo designs may need to be revised based on the actual signal directions surfaced here.